In [7]:
import pandas as pd, numpy as np, warnings; warnings.filterwarnings('ignore')

s = pd.read_csv('sales_transactions_cleaned.csv')
p = pd.read_csv('products.csv')
s['revenue'] =  (s['quantity']*s['price'])- pd.to_numeric(s['discount_amount'],errors='coerce').fillna(0)
s['date'] = pd.to_datetime(s['date'])
s['month'] = s['date'].dt.to_period('M').astype(str)
r''
clean = lambda col: pd.to_numeric(col.astype(str).str.replace(r'[^0-9.\-]','',regex=True), errors='coerce').abs()
p['cost_c'] = clean(p['cost'])

perf = (s.groupby('product_id').agg(total_quantity_sold=('quantity','sum'), total_revenue=('revenue','sum'))
       .reset_index()
       .merge(p[['product_id','cost_c']], on='product_id', how='left').assign(total_cost= lambda x: x['total_quantity_sold'] * x['cost_c'],
                                                                              profit_margin = lambda x: ((x['total_revenue']-x['total_quantity_sold'] * x['cost_c']) / x['total_revenue']).round(4))
        [['product_id','total_quantity_sold','total_revenue','profit_margin']].sort_values('total_revenue',ascending=False).round(2))

print(perf)


    product_id  total_quantity_sold  total_revenue  profit_margin
11          15                11965       55457.02           0.84
3            6                 7064       32487.58           0.67
17          21                 6801       31045.62           0.69
2            5                 6600       30268.71           0.56
6            9                   10          45.81           0.56
1            3                    9          37.61           0.52
14          18                    6          35.82           0.83
16          20                    5          31.75           0.84
15          19                    2          10.12           0.74
0            2                    3           9.81           0.39
8           12                    1           7.85           0.81
7           11                    1           7.10           0.72
10          14                    1           5.25           0.47
5            8                    1           4.36           0.82
12        

In [8]:
m = (s.groupby(['product_id','month']).agg(qty = ('quantity','sum'), avg_price=('price','mean'))
    .reset_index().sort_values(['product_id','month']))
m['ped'] = (m.groupby('product_id')['qty'].pct_change() /
           m.groupby('product_id')['avg_price'].pct_change()).replace([np.inf,-np.inf],np.nan)
ped = (m.groupby('product_id')['ped'].mean().reset_index().round(4)
      .rename(columns={'ped' : 'price_elasticity_of_demand'})
      .assign(suggested_price_change = lambda x: x['price_elasticity_of_demand'].apply(lambda v: '-5%' if abs(v)>1 else '+5%' if pd.notna(v) else '0%' )))

print(ped)

    product_id  price_elasticity_of_demand suggested_price_change
0            2                         NaN                     0%
1            3                    -30.1452                    -5%
2            5                     25.7223                    -5%
3            6                     88.0175                    -5%
4            7                         NaN                     0%
5            8                         NaN                     0%
6            9                      1.0103                    -5%
7           11                         NaN                     0%
8           12                         NaN                     0%
9           13                         NaN                     0%
10          14                         NaN                     0%
11          15                     75.7618                    -5%
12          16                         NaN                     0%
13          17                         NaN                     0%
14        

In [9]:
perf.to_csv('Session5_Product_PerformanceVTest.csv', index=False)
ped.to_csv('Session5_Price_AnalysisVTest.csv', index=False)
print('Success')

Success
